In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols, logit, probit
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')
SwissLabor = pd.read_csv('../datasets/SwissLabor.csv')

# Предобработка SwissLabor
SwissLabor['participation'] = SwissLabor['participation'].replace({'yes': 1, 'no': 0}).astype(int)
SwissLabor['foreign'] = SwissLabor['foreign'].replace({'yes': 1, 'no': 0}).astype(int)

# Удаление пропусков для loanapp, чтобы размер выборки был одинаковым во всех моделях
loanapp = loanapp[['approve', 'appinc', 'mortno', 'unem', 'dep', 'male']].dropna()

my_digits = 3


# labour force equation

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, np.log(FAMINC), WE, KL6, K618, CIT, UN__
трёх спецификаций:
- LPM
- logit
- probit

Результаты подгонки:

In [ ]:
#| echo: false
#| warning: false
#| output: asis

f_lfp = 'LFP ~ 1 + WA + np.log(FAMINC) + WE + KL6 + K618 + CIT + UN'

regr_LPM_lfp = ols(f_lfp, data=mroz_Greene).fit()
regr_logit_lfp = logit(f_lfp, data=mroz_Greene).fit(disp=0)
regr_probit_lfp = probit(f_lfp, data=mroz_Greene).fit(disp=0)

models_lfp = [regr_LPM_lfp, regr_logit_lfp, regr_probit_lfp]
model_names_lfp = ['LPM', 'logit', 'probit']

compare_table_lfp = summary_col(
    models_lfp, 
    model_names=model_names_lfp,
    stars=True, 
    regressor_order=['WA', 'np.log(FAMINC)', 'WE', 'KL6', 'K618', 'CIT', 'UN', 'Intercept'],
    float_format=f'%0.{my_digits}f'
)

df_comp_lfp = compare_table_lfp.tables[0]

# Добавление строки с зависимой переменной
dep_vars_dict_lfp = {col: [m.model.endog_names] for col, m in zip(df_comp_lfp.columns, models_lfp)}
top_row_lfp = pd.DataFrame(dep_vars_dict_lfp, index=['Dep. Variable'])
df_comp_lfp = pd.concat([top_row_lfp, df_comp_lfp])

legend_text = "\n\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"
display(Markdown(df_comp_lfp.to_markdown() + legend_text))

|                | LPM       | logit     | probit    |
|:---------------|:----------|:----------|:----------|
| Dep. Variable  | LFP       | LFP       | LFP       |
| WA             | -0.013*** | -0.063*** | -0.038*** |
|                | (0.003)   | (0.013)   | (0.008)   |
| np.log(FAMINC) | 0.075**   | 0.341**   | 0.205**   |
|                | (0.037)   | (0.172)   | (0.104)   |
| WE             | 0.038***  | 0.179***  | 0.108***  |
|                | (0.008)   | (0.040)   | (0.024)   |
| KL6            | -0.302*** | -1.443*** | -0.868*** |
|                | (0.036)   | (0.194)   | (0.112)   |
| K618           | -0.018    | -0.095    | -0.057    |
|                | (0.014)   | (0.067)   | (0.040)   |
| CIT            | -0.048    | -0.214    | -0.126    |
|                | (0.038)   | (0.176)   | (0.107)   |
| UN             | -0.004    | -0.017    | -0.011    |
|                | (0.006)   | (0.026)   | (0.016)   |
| Intercept      | 0.079     | -1.856    | -1.108    |
|                | (0.356)   | (1.679)   | (1.014)   |
| R-squared      | 0.130     |           |           |
| R-squared Adj. | 0.122     |           |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Рассмотрим несколько человек с характеристиками. Постройте прогноз для каждого. **Ответ округлите до 3-х десятичных знаков.**

In [3]:
#| echo: false
#| warning: false
#| output: asis

new_df_lfp = pd.DataFrame({
    'WA': [35, 40, 42], 
    'FAMINC': [12500, 9800, 67800], 
    'WE': [15, 12, 14], 
    'KL6': [2, 1, 2], 
    'K618': [0, 2, 1], 
    'CIT': [1, 0, 1], 
    'UN': [5, 7.5, 3]
})

display(Markdown("**Характеристики:**\n\n" + new_df_lfp.to_markdown(index=False) + "\n"))

df_predict_lfp = pd.DataFrame({
    'Прогноз LPM': regr_LPM_lfp.predict(new_df_lfp),
    'Прогноз logit': regr_logit_lfp.predict(new_df_lfp),
    'Прогноз probit': regr_probit_lfp.predict(new_df_lfp)
})

df_predict_lfp = df_predict_lfp.round(my_digits)
df_predict_lfp.index = df_predict_lfp.index + 1
df_predict_lfp.index.name = 'Человек'

display(Markdown("**Ответ:**\n\n" + df_predict_lfp.to_markdown(index=True)))

**Характеристики:**

|   WA |   FAMINC |   WE |   KL6 |   K618 |   CIT |   UN |
|-----:|---------:|-----:|------:|-------:|------:|-----:|
|   35 |    12500 |   15 |     2 |      0 |     1 |  5   |
|   40 |     9800 |   12 |     1 |      2 |     0 |  7.5 |
|   42 |    67800 |   14 |     2 |      1 |     1 |  3   |


**Ответ:**

|   Человек |   Прогноз LPM |   Прогноз logit |   Прогноз probit |
|----------:|--------------:|----------------:|-----------------:|
|         1 |         0.221 |           0.209 |            0.214 |
|         2 |         0.329 |           0.3   |            0.307 |
|         3 |         0.207 |           0.192 |            0.196 |

# approve equation

Для датасета `loanapp` рассмотрим регрессию __approve на I(appinc / 100), mortno, unem, dep, male__ 
трёх спецификаций:
- LPM
- logit
- probit

Результаты подгонки:

In [ ]:
#| echo: false
#| warning: false
#| output: asis

f_app = 'approve ~ 1 + I(appinc/100) + mortno + unem + dep + male'

regr_LPM_app = ols(f_app, data=loanapp).fit()
regr_logit_app = logit(f_app, data=loanapp).fit(disp=0)
regr_probit_app = probit(f_app, data=loanapp).fit(disp=0)

models_app = [regr_LPM_app, regr_logit_app, regr_probit_app]
model_names_app = ['LPM', 'logit', 'probit']

compare_table_app = summary_col(
    models_app, 
    model_names=model_names_app,
    stars=True, 
    regressor_order=['I(appinc / 100)', 'mortno', 'unem', 'dep', 'male', 'Intercept'],
    float_format=f'%0.{my_digits}f'
)

df_comp_app = compare_table_app.tables[0]

# Добавление строки с зависимой переменной
dep_vars_dict_app = {col: [m.model.endog_names] for col, m in zip(df_comp_app.columns, models_app)}
top_row_app = pd.DataFrame(dep_vars_dict_app, index=['Dep. Variable'])
df_comp_app = pd.concat([top_row_app, df_comp_app])

display(Markdown(df_comp_app.to_markdown() + legend_text))

|                 | LPM      | logit    | probit   |
|:----------------|:---------|:---------|:---------|
| Dep. Variable   | approve  | approve  | approve  |
| I(appinc / 100) | -0.014*  | -0.106   | -0.056   |
|                 | (0.009)  | (0.067)  | (0.038)  |
| mortno          | 0.079*** | 0.817*** | 0.422*** |
|                 | (0.016)  | (0.170)  | (0.086)  |
| unem            | -0.008** | -0.065** | -0.036** |
|                 | (0.003)  | (0.029)  | (0.016)  |
| dep             | -0.012*  | -0.106*  | -0.055*  |
|                 | (0.007)  | (0.061)  | (0.033)  |
| male            | 0.019    | 0.173    | 0.093    |
|                 | (0.019)  | (0.176)  | (0.094)  |
| Intercept       | 0.886*** | 2.032*** | 1.198*** |
|                 | (0.022)  | (0.193)  | (0.105)  |
| R-squared       | 0.017    |          |          |
| R-squared Adj.  | 0.014    |          |          |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Рассмотрим несколько человек с характеристиками. Постройте прогноз для каждого. **Ответ округлите до 3-х десятичных знаков.**

In [5]:
#| echo: false
#| warning: false
#| output: asis

new_df_app = pd.DataFrame({
    'appinc': [120, 48, 82], 
    'mortno': [1, 1, 0], 
    'unem': [1.8, 3.2, 3.9], 
    'dep': [0, 0, 1], 
    'male': [1, 0, 1]
})

display(Markdown("**Характеристики:**\n\n" + new_df_app.to_markdown(index=False) + "\n"))

df_predict_app = pd.DataFrame({
    'Прогноз LPM': regr_LPM_app.predict(new_df_app),
    'Прогноз logit': regr_logit_app.predict(new_df_app),
    'Прогноз probit': regr_probit_app.predict(new_df_app)
})

df_predict_app = df_predict_app.round(my_digits)
df_predict_app.index = df_predict_app.index + 1
df_predict_app.index.name = 'Человек'

display(Markdown("**Ответ:**\n\n" + df_predict_app.to_markdown(index=True)))

**Характеристики:**

|   appinc |   mortno |   unem |   dep |   male |
|---------:|---------:|-------:|------:|-------:|
|      120 |        1 |    1.8 |     0 |      1 |
|       48 |        1 |    3.2 |     0 |      0 |
|       82 |        0 |    3.9 |     1 |      1 |


**Ответ:**

|   Человек |   Прогноз LPM |   Прогноз logit |   Прогноз probit |
|----------:|--------------:|----------------:|-----------------:|
|         1 |         0.953 |           0.941 |            0.943 |
|         2 |         0.933 |           0.93  |            0.93  |
|         3 |         0.851 |           0.853 |            0.853 |

# swiss labour force equation

Для датасета `SwissLabor` рассмотрим регрессию __participation на income, I(income ** 2), age, I(age ** 2), youngkids, oldkids__
трёх спецификаций:
- LPM
- logit
- probit

Замечание: нужно перекодировать столбцы __participation и foreign__
Результаты подгонки:

In [6]:
#| echo: false
#| warning: false
#| output: asis

f_swiss = 'participation ~ income + I(income**2) + age + I(age**2) + youngkids + oldkids'

regr_LPM_swiss = ols(f_swiss, data=SwissLabor).fit()
regr_logit_swiss = logit(f_swiss, data=SwissLabor).fit(disp=0)
regr_probit_swiss = probit(f_swiss, data=SwissLabor).fit(disp=0)

models_swiss = [regr_LPM_swiss, regr_logit_swiss, regr_probit_swiss]
model_names_swiss = ['LPM', 'logit', 'probit']

compare_table_swiss = summary_col(
    models_swiss, 
    model_names=model_names_swiss,
    stars=True, 
    regressor_order=['income', 'I(income ** 2)', 'age', 'I(age ** 2)', 'youngkids', 'oldkids', 'Intercept'],
    float_format=f'%0.{my_digits}f'
)

df_comp_swiss = compare_table_swiss.tables[0]

# Добавление строки с зависимой переменной
dep_vars_dict_swiss = {col: [m.model.endog_names] for col, m in zip(df_comp_swiss.columns, models_swiss)}
top_row_swiss = pd.DataFrame(dep_vars_dict_swiss, index=['Dep. Variable'])
df_comp_swiss = pd.concat([top_row_swiss, df_comp_swiss])

display(Markdown(df_comp_swiss.to_markdown() + legend_text))

|                | LPM           | logit         | probit        |
|:---------------|:--------------|:--------------|:--------------|
| Dep. Variable  | participation | participation | participation |
| income         | 0.269         | 3.146         | 1.973         |
|                | (0.654)       | (3.400)       | (2.090)       |
| I(income ** 2) | -0.025        | -0.211        | -0.131        |
|                | (0.031)       | (0.161)       | (0.099)       |
| age            | 0.790***      | 3.875***      | 2.347***      |
|                | (0.131)       | (0.669)       | (0.397)       |
| I(age ** 2)    | -0.111***     | -0.544***     | -0.329***     |
|                | (0.016)       | (0.083)       | (0.049)       |
| youngkids      | -0.227***     | -1.065***     | -0.637***     |
|                | (0.032)       | (0.166)       | (0.096)       |
| oldkids        | -0.052***     | -0.250***     | -0.151***     |
|                | (0.017)       | (0.083)       | (0.050)       |
| Intercept      | -0.716        | -15.282       | -9.643        |
|                | (3.488)       | (17.960)      | (11.066)      |
| R-squared      | 0.155         |               |               |
| R-squared Adj. | 0.149         |               |               |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Рассмотрим несколько человек с характеристиками. Постройте прогноз для каждого. **Ответ округлите до 3-х десятичных знаков.**

In [7]:
#| echo: false
#| warning: false
#| output: asis

new_df_swiss = pd.DataFrame({
    'income': [11.367, 9.217, 10.686], 
    'age': [2.5, 3.7, 4.2], 
    'youngkids': [0, 2, 2], 
    'oldkids': [0, 0, 1]
})

display(Markdown("**Характеристики:**\n\n" + new_df_swiss.to_markdown(index=False) + "\n"))

df_predict_swiss = pd.DataFrame({
    'Прогноз LPM': regr_LPM_swiss.predict(new_df_swiss),
    'Прогноз logit': regr_logit_swiss.predict(new_df_swiss),
    'Прогноз probit': regr_probit_swiss.predict(new_df_swiss)
})

df_predict_swiss = df_predict_swiss.round(my_digits)
df_predict_swiss.index = df_predict_swiss.index + 1
df_predict_swiss.index.name = 'Человек'

display(Markdown("**Ответ:**\n\n" + df_predict_swiss.to_markdown(index=True)))

**Характеристики:**

|   income |   age |   youngkids |   oldkids |
|---------:|------:|------------:|----------:|
|   11.367 |   2.5 |           0 |         0 |
|    9.217 |   3.7 |           2 |         0 |
|   10.686 |   4.2 |           2 |         1 |


**Ответ:**

|   Человек |   Прогноз LPM |   Прогноз logit |   Прогноз probit |
|----------:|--------------:|----------------:|-----------------:|
|         1 |         0.419 |           0.367 |            0.371 |
|         2 |         0.606 |           0.626 |            0.624 |
|         3 |         0.18  |           0.182 |            0.186 |